## 1) Setup Kredensial Kaggle

Notebook ini diawali dengan menyiapkan kredensial Kaggle agar proses pengunduhan dataset kompetisi dapat dilakukan secara otomatis dari runtime. Tahap ini penting karena seluruh pipeline cleaning bergantung pada keberhasilan akses data sumber sejak awal.

Pada konteks notebook kompetisi, tahap setup seperti ini berfungsi sebagai **fondasi reprodusibilitas**: sebelum analisis dilakukan, sumber data harus dapat diakses dengan cara yang konsisten dan terkontrol. Dengan demikian, seluruh proses berikutnya—mulai dari validasi struktur dataset hingga audit anomali—berjalan di atas data yang sama.

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = "-"
os.environ['KAGGLE_KEY'] = "-"

Tahap ini menyiapkan *environment variable* yang dibutuhkan untuk autentikasi Kaggle sehingga proses unduh dataset dapat dijalankan tanpa perlu menulis ulang kredensial di banyak bagian notebook. Secara praktis, langkah ini membuat alur kerja lebih rapi, lebih mudah dipelihara, dan mengurangi risiko kesalahan konfigurasi saat notebook dijalankan ulang.

## 2) Definisi Fungsi Download Dataset

In [ ]:
def download_competition_data():
    print("Mulai mengunduh dataset dari Kaggle...")
    path = kagglehub.competition_download('data-analytics-competition-dac-find-it-2026')
    print(f"  Data berhasil diunduh dan disimpan di: {path}")
    return path

Fungsi download dibungkus ke dalam satu utilitas khusus agar proses pengambilan data menjadi **reusable** dan lebih terstruktur. Pendekatan ini memudahkan notebook untuk dijalankan ulang dari awal tanpa perlu mengulang logika unduh secara manual, serta membuat alur persiapan data lebih bersih sebelum masuk ke tahap eksplorasi dan cleaning.

## 3) Import Library Inti dan Inisialisasi Path

In [ ]:
import os
import kagglehub
import pandas as pd
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def download_competition_data():
    print("Mulai mengunduh dataset dari Kaggle...")
    path = kagglehub.competition_download('data-analytics-competition-dac-find-it-2026')
    print(f"Data berhasil diunduh dan disimpan di: {path}")
    return path

path_root = download_competition_data()

train_dir = os.path.join(path_root, 'train')
test_dir = os.path.join(path_root, 'test')
csv_path = os.path.join(path_root, 'samplesubmission.csv')

Pada tahap ini, notebook memuat dependensi utama, menjalankan proses pengunduhan dataset, dan menetapkan path penting seperti folder `train`, `test`, serta file `samplesubmission.csv`. Langkah ini diperlukan agar seluruh tahapan berikutnya memiliki referensi direktori yang konsisten.

Output pada bagian ini menunjukkan bahwa dataset berhasil diunduh dan diekstrak. Ini menjadi titik awal yang penting karena analisis distribusi kelas, inspeksi visual, dan proses cleaning hanya dapat dilakukan apabila struktur file telah tersedia dengan benar di runtime.

## 4) Validasi Struktur Dataset Awal

In [ ]:
import os
from collections import Counter
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

print('Path root :', path_root)
print('Train dir :', train_dir, '| exists =', os.path.exists(train_dir))
print('Test dir  :', test_dir,  '| exists =', os.path.exists(test_dir))
print('CSV path  :', csv_path,  '| exists =', os.path.exists(csv_path))

train_classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
print('\nJumlah kelas pada train:', len(train_classes))
print('Daftar kelas:', train_classes[:10])

num_test_images = len([f for f in os.listdir(test_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))])
print('Jumlah gambar pada test:', num_test_images)

Sebelum masuk ke analisis yang lebih dalam, dilakukan **sanity check** untuk memastikan bahwa struktur dataset memang sesuai dengan ekspektasi task kompetisi. Validasi ini mencakup pengecekan keberadaan folder `train`, `test`, dan file submission, sekaligus memastikan jumlah kelas pada data train serta jumlah gambar pada data test.

Hasil validasi menunjukkan bahwa seluruh path utama **terdeteksi dengan benar**, folder train berisi **6 kelas**, dan folder test memuat **404 gambar**. Temuan ini penting karena menegaskan bahwa dataset telah termuat secara utuh dan notebook dapat melanjutkan proses eksplorasi tanpa risiko error akibat path atau struktur file yang tidak valid.

## 5) Hitung Distribusi Gambar per Kelas

In [ ]:
class_counts = {}
for c in train_classes:
    c_dir = os.path.join(train_dir, c)
    n = len([f for f in os.listdir(c_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))])
    class_counts[c] = n

counts_series = pd.Series(class_counts).sort_values(ascending=False)
print('\nRingkasan jumlah gambar per kelas:')
print(counts_series.describe())
print('\nJumlah masing masing kelas:')
print(counts_series.head(5))


plt.figure(figsize=(12, 5))
counts_series.plot(kind='bar')
plt.title('Distribusi Jumlah Gambar per Kelas (Train)')
plt.xlabel('Kelas')
plt.ylabel('Jumlah Gambar')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

Distribusi kelas dianalisis lebih awal untuk memahami kondisi mentah dataset sebelum cleaning dilakukan. Langkah ini penting karena **ketimpangan jumlah sampel antar kelas** dapat memengaruhi strategi pembersihan data sekaligus perilaku model saat training.

### Ringkasan Distribusi Awal

| Kelas | Jumlah Gambar |
|---|---:|
| realperson | 465 |
| fake_unknown | 338 |
| fake_mask | 278 |
| fake_screen | 229 |
| fake_mannequin | 224 |
| fake_printed | 118 |

### Statistik Singkat

| Metrik | Nilai |
|---|---:|
| Total kelas | 6 |
| Total gambar | 1652 |
| Rata-rata per kelas | 275.33 |
| Standar deviasi | 117.85 |
| Minimum | 118 |
| Maksimum | 465 |

Bar chart memperlihatkan bahwa **`realperson` merupakan kelas terbesar**, sedangkan **`fake_printed` merupakan kelas terkecil**. Selisih antara keduanya cukup besar, sehingga dataset mentah belum sepenuhnya seimbang.

📊 **Interpretasi utama:** terdapat **class imbalance** yang nyata sejak awal. Kondisi ini perlu diingat selama cleaning agar penghapusan data tidak memperburuk ketimpangan distribusi.

## 6) Visual Sampling Gambar per Kelas

Setelah melihat distribusi numerik, langkah berikutnya adalah melakukan **inspeksi visual cepat** terhadap sampel dari setiap kelas. Tujuan tahap ini adalah memahami karakter visual masing-masing kelas, mendeteksi kemungkinan **noise label**, kualitas gambar yang tidak konsisten, serta melihat apakah ada pola yang berpotensi menyulitkan proses klasifikasi.

Dari grid sampel yang ditampilkan, terlihat bahwa beberapa kelas memiliki ciri yang relatif jelas, tetapi sebagian lainnya menunjukkan variasi visual yang tinggi. Kelas seperti `fake_unknown` tampak paling heterogen, sedangkan beberapa kelas lain masih memperlihatkan batas visual yang cukup mudah dikenali. Hasil inspeksi ini penting karena mengindikasikan bahwa masalah dataset bukan hanya soal jumlah, tetapi juga soal **keragaman intra-kelas** dan kemungkinan kontaminasi label.

🔍 Dengan kata lain, tahap ini memberi konteks visual sebelum notebook beralih ke pendekatan yang lebih kuantitatif melalui embedding, similarity, dan audit anomali.

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

num_classes_to_show = 6
num_images_per_class = 10

selected_classes = train_classes[:num_classes_to_show]

fig, axes = plt.subplots(len(selected_classes), num_images_per_class,
                         figsize=(20, 4 * len(selected_classes)))

if len(selected_classes) == 1:
    axes = [axes]

for i, cls in enumerate(selected_classes):
    cls_dir = os.path.join(train_dir, cls)
    all_images = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))]
    num_samples = min(num_images_per_class, len(all_images))
    random_images = random.sample(all_images, num_samples)

    for j in range(num_images_per_class):
        if len(selected_classes) > 1:
            ax = axes[i, j]
        else:
            ax = axes[j]

        if j < num_samples:
            filename = random_images[j]
            img_path = os.path.join(cls_dir, filename)
            img = mpimg.imread(img_path)
            ax.imshow(img)

            # Menambahkan nama file di bawah gambar
            ax.set_xlabel(filename, fontsize=8)
            ax.xaxis.set_label_position('bottom')
            ax.set_xticks([])
            ax.set_yticks([])

            if j == 0:
                ax.set_ylabel(cls, fontsize=14, fontweight='bold', rotation=0, labelpad=80, verticalalignment='center')
        else:
            ax.axis('off')

plt.tight_layout()
plt.show()

## 7) Ekstraksi Embedding DINOv3

Agar proses cleaning tidak hanya bergantung pada inspeksi manual, seluruh gambar diubah menjadi **embedding visual** menggunakan model **DINOv3**. Embedding adalah representasi numerik yang merangkum karakter visual gambar ke dalam vektor fitur, sehingga hubungan kemiripan antar gambar dapat diukur secara kuantitatif.

Output menunjukkan bahwa seluruh **1.652 gambar** berhasil diproses, lalu dinormalisasi, dan menghasilkan embedding dengan bentuk akhir **`(1652, 1024)`**. Artinya, setiap gambar direpresentasikan sebagai vektor berdimensi 1024 yang siap digunakan untuk perhitungan cosine similarity.

✅ **Makna tahap ini:** karena jumlah embedding yang dihasilkan sama dengan jumlah total gambar, tidak ada indikasi kegagalan baca file pada tahap ekstraksi fitur. Ini sangat penting, sebab seluruh proses cleaning lanjutan—terutama deteksi duplikat dan audit anomali—akan bergantung pada kualitas representasi embedding ini.

In [ ]:
import os
import numpy as np
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
from huggingface_hub import login
from sklearn.preprocessing import normalize

login(token="-")

model_name = "facebook/dinov3-vitl16-pretrain-lvd1689m"

print(f"\nLoading model: {model_name}")
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

torch.backends.cuda.matmul.allow_tf32 = True

# VALIDASI DATASET (
print("\n=== DATASET INFO ===")
print("Train dir:", train_dir)
print("Jumlah kelas:", len(train_classes))

total_images = 0
class_counts = {}

all_image_paths = []

for c in train_classes:
    c_dir = os.path.join(train_dir, c)

    if not os.path.exists(c_dir):
        print(f"Folder tidak ditemukan: {c_dir}")
        continue

    valid_images = [
        os.path.join(c_dir, f)
        for f in os.listdir(c_dir)
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))
    ]

    class_counts[c] = len(valid_images)
    total_images += len(valid_images)

    all_image_paths.extend(valid_images)

print("\nTotal gambar:", total_images)
print("Contoh distribusi class:", list(class_counts.items())[:5])

# EMBEDDING FUNCTION
def extract_embedding_batch(img_paths, batch_size=8):
    embeddings = []
    valid_paths = []

    for i in range(0, len(img_paths), batch_size):
        batch_paths = img_paths[i:i+batch_size]

        images = []
        temp_paths = []

        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img)
                temp_paths.append(p)
            except Exception as e:
                print(f"Skip {p}: {e}")

        if len(images) == 0:
            continue

        inputs = processor(images=images, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        batch_emb = outputs.pooler_output.cpu().numpy()

        embeddings.extend(batch_emb)
        valid_paths.extend(temp_paths)

        print(f"Processed {min(i+batch_size, len(img_paths))}/{len(img_paths)}")

    return np.array(embeddings), valid_paths

# EXTRACT EMBEDDING
print("\n=== EXTRACT EMBEDDING ===")
embeddings, valid_paths = extract_embedding_batch(
    all_image_paths,
    batch_size=8
)

# NORMALIZATION
embeddings = normalize(embeddings)

print("\nDone!")
print("Shape embedding:", embeddings.shape)

## 8) Deteksi Grup Duplikasi dengan Cosine Similarity

Setelah embedding tersedia, notebook mencari gambar-gambar yang **sangat mirip** menggunakan **cosine similarity**. Tujuan utamanya adalah mengidentifikasi **redundansi data** yang dapat menurunkan kualitas dataset, terutama bila gambar yang sama atau nyaris sama muncul berulang kali.

Hasil deteksi menunjukkan adanya **199 grup gambar duplikat**. Jumlah ini cukup besar dan menunjukkan bahwa masalah duplikasi pada dataset bukan kasus kecil atau sporadis. Dengan threshold similarity yang ketat, grup-grup yang terdeteksi cenderung berisi gambar yang memang sangat dekat secara visual.

📌 **Interpretasi penting:** keberadaan ratusan grup duplikat menandakan bahwa dataset mentah mengandung banyak contoh yang tidak benar-benar independen. Dalam konteks kompetisi, kondisi seperti ini dapat meningkatkan noise, memperbesar risiko *overfitting* terhadap pola berulang, dan menurunkan kualitas sinyal pembelajaran yang diterima model.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(embeddings)

threshold = 0.97
visited = set()
duplicate_groups = []

for i in range(len(valid_paths)):
    if i in visited:
        continue

    cluster_indices = []
    queue = [i]
    visited.add(i)

    while queue:
        current_idx = queue.pop(0)
        cluster_indices.append(current_idx)

        for j in range(len(valid_paths)):
            if j not in visited and similarity_matrix[current_idx, j] >= threshold:
                visited.add(j)
                queue.append(j)

    if len(cluster_indices) > 1:
        group_paths = [valid_paths[idx] for idx in cluster_indices]
        duplicate_groups.append(group_paths)

print("\n--- Hasil Deteksi Grup Duplikasi ---")
if len(duplicate_groups) == 0:
    print(f"Tidak ditemukan gambar duplikat dengan threshold {threshold}.")
else:
    print(f"Ditemukan {len(duplicate_groups)} grup gambar duplikat:")
    for idx, group in enumerate(duplicate_groups[:10]):
        print(f"\nGrup {idx + 1} ({len(group)} gambar terdeteksi mirip):")
        for img_path in group:
            print(f" - {img_path}")

## 9) Visualisasi Hasil Grup Duplikat

Deteksi duplikat secara numerik perlu diverifikasi secara visual agar proses cleaning tidak menjadi sekadar penyaringan otomatis tanpa interpretasi. Oleh karena itu, notebook menampilkan beberapa grup duplikat teratas beserta anggota grupnya untuk memastikan bahwa similarity tinggi benar-benar merepresentasikan duplikasi yang relevan.

### Ringkasan Temuan Duplikat

| Temuan | Makna |
|---|---|
| Gambar identik dalam satu grup | Menunjukkan redundansi nyata dalam dataset |
| Gambar identik muncul di kelas berbeda | Mengindikasikan konflik label lintas kelas |
| Similarity sangat tinggi / 100% | Bukti kuat bahwa kasus tersebut layak diprioritaskan untuk cleaning |

Dari visualisasi ini, terlihat bahwa beberapa grup memang berisi gambar yang **identik atau nyaris identik**, bahkan pada beberapa kasus tersebar ke **kelas yang berbeda**. Temuan seperti ini sangat penting karena menunjukkan bahwa masalah pada dataset bukan hanya duplikasi dalam kelas yang sama, tetapi juga **duplikasi lintas label** yang berpotensi mengajarkan sinyal kelas yang saling bertentangan.

⚠️ Dengan demikian, visualisasi ini menjadi bukti kuat bahwa deduplikasi bukan hanya langkah perapian dataset, tetapi juga langkah untuk meningkatkan **konsistensi label** dan kualitas data latih.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

groups_to_show = duplicate_groups[:10]

if len(groups_to_show) == 0:
    print("Tidak ada grup duplikat untuk divisualisasikan.")
else:
    print(f"\nMenampilkan visualisasi untuk {len(groups_to_show)} grup teratas:")

    for i, group in enumerate(groups_to_show):
        num_images = len(group)
        display_count = min(num_images, 6)

        fig, axes = plt.subplots(1, display_count, figsize=(4 * display_count, 4))

        fig.suptitle(f"Grup {i + 1} ({num_images} Gambar Duplikat)", fontsize=14, fontweight='bold', y=1.05)


        if display_count == 1:
            axes = [axes]


        anchor_path = group[0]

        anchor_idx = valid_paths.index(anchor_path)

        for j in range(display_count):
            img_path = group[j]
            img = mpimg.imread(img_path)


            current_idx = valid_paths.index(img_path)

            sim_score = similarity_matrix[anchor_idx, current_idx] * 100

            axes[j].imshow(img)


            title_text = f"{os.path.basename(img_path)}\nKemiripan: {sim_score:.1f}%"


            if j == 0:
                axes[j].set_title(title_text, fontsize=10, color='blue', fontweight='bold')
            else:
                axes[j].set_title(title_text, fontsize=10)

            axes[j].axis('off')

        plt.tight_layout()
        plt.show()

## 10) Deteksi Anomali Label dengan Centroid + Rekomendasi Kelas

Selain duplikasi, notebook juga mengaudit kemungkinan **salah label** dengan pendekatan berbasis centroid kelas. Setiap gambar dibandingkan dengan centroid kelas asalnya, lalu gambar dengan similarity rendah diperlakukan sebagai kandidat anomali. Notebook juga menampilkan rekomendasi kelas alternatif berdasarkan centroid yang paling mirip.

### Ringkasan Audit Awal

| Komponen | Nilai |
|---|---:|
| Threshold similarity | 0.7 |
| Total gambar train awal | 1652 |
| Total kandidat anomali | 1396 |
| Persentase kandidat anomali | 84.5% |

Dengan threshold **0,7**, ditemukan **1.396 gambar** yang tergolong anomali. Jumlah ini sangat besar, sehingga hasilnya lebih tepat dibaca sebagai **screening awal untuk kandidat review**, bukan sebagai keputusan otomatis bahwa seluruh gambar tersebut pasti salah label.

📌 **Interpretasi metodologis:** output sebesar ini menunjukkan bahwa beberapa kelas memiliki variasi visual yang tinggi, dan threshold yang digunakan cukup sensitif. Karena itu, tahap ini sangat berguna untuk mempersempit area audit, tetapi tetap perlu divalidasi melalui inspeksi visual.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.metrics.pairwise import cosine_similarity

print("Mengelompokkan data berdasarkan kelas...")
class_data = {c: {'paths': [], 'embeddings': []} for c in train_classes}

for path, emb in zip(valid_paths, embeddings):
    class_name = os.path.basename(os.path.dirname(path))
    if class_name in class_data:
        class_data[class_name]['paths'].append(path)
        class_data[class_name]['embeddings'].append(emb)


print("Menghitung centroid untuk semua kelas...")
class_centroids = {}
for c in train_classes:
    embs = class_data[c]['embeddings']
    if len(embs) > 0:

        class_centroids[c] = np.mean(np.array(embs), axis=0).reshape(1, -1)

anomalies_per_class = {}
total_anomalies_detected = 0
anomaly_threshold = 0.7

print(f"Menghitung jarak anomali dan mencari rekomendasi kelas (Threshold < {anomaly_threshold})...")
for c in train_classes:
    paths = class_data[c]['paths']
    embs = class_data[c]['embeddings']

    if len(paths) < 3 or c not in class_centroids:
        continue

    embs_array = np.array(embs)
    centroid = class_centroids[c]
    similarities = cosine_similarity(embs_array, centroid).flatten()

    anomalous_items = []


    for i in range(len(paths)):
        sim_to_own_class = similarities[i]


        if sim_to_own_class < anomaly_threshold:
            img_emb = embs_array[i].reshape(1, -1)

            best_suggested_class = "Tidak Diketahui"
            best_suggested_sim = -1


            for other_class, other_centroid in class_centroids.items():
                if other_class == c:
                    continue


                sim_to_other = cosine_similarity(img_emb, other_centroid)[0][0]

                if sim_to_other > best_suggested_sim:
                    best_suggested_sim = sim_to_other
                    best_suggested_class = other_class


            anomalous_items.append((paths[i], sim_to_own_class, best_suggested_class, best_suggested_sim))


    anomalous_items.sort(key=lambda x: x[1])

    if len(anomalous_items) > 0:
        anomalies_per_class[c] = anomalous_items
        total_anomalies_detected += len(anomalous_items)


print("\n" + "="*40)
print(f"TOTAL ANOMALI DITEMUKAN: {total_anomalies_detected} gambar")
print("="*40)



for c, anomalies in anomalies_per_class.items():
    num_anomalies = len(anomalies)

    cols = 5
    rows = (num_anomalies + cols - 1) // cols


    fig, axes = plt.subplots(rows, cols, figsize=(15, 4.5 * rows))
    fig.suptitle(f"\nAnomali di Kelas: '{c}' ({num_anomalies} gambar)", fontsize=16, fontweight='bold', y=1.02)

    if type(axes) is np.ndarray:
        axes = axes.flatten()
    else:
        axes = [axes]

    for j in range(len(axes)):
        if j < num_anomalies:

            img_path, sim_own, suggested_class, sim_suggested = anomalies[j]
            img = mpimg.imread(img_path)
            axes[j].imshow(img)


            title_text = (
                f"{os.path.basename(img_path)}\n"
                f"Asli: {sim_own:.3f}\n"
                f"-> {suggested_class}: {sim_suggested:.3f}"
            )


            if sim_suggested > 0.8:
                text_color = 'green'
            else:
                text_color = 'red'

            axes[j].set_title(title_text, fontsize=9, color=text_color, fontweight='bold')

        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

## 11) Hapus Duplikat Berdasarkan Skor Kemiripan ke Centroid Kelas

Pada setiap grup duplikat, notebook tidak menghapus file secara acak. Sebaliknya, satu gambar dipertahankan berdasarkan **skor kemiripan tertinggi terhadap centroid kelasnya**, sedangkan anggota lain di dalam grup dihapus. Strategi ini lebih kuat dibanding menyimpan file pertama secara sembarang, karena keputusan penghapusan dibuat berdasarkan seberapa representatif gambar tersebut terhadap labelnya saat ini.

### Aturan Keputusan Cleaning

| Langkah | Keputusan |
|---|---|
| Temukan grup gambar yang sangat mirip | Bentuk grup duplikat |
| Hitung similarity tiap anggota ke centroid kelas | Ukur representativitas |
| Pilih skor tertinggi | Gambar dipertahankan |
| Hapus anggota lain dalam grup | Kurangi redundansi dan konflik label |

Pada akhir proses, tercatat bahwa **225 gambar duplikat berhasil dihapus**.

✅ **Makna tahap ini:** filtering ini tidak hanya mengurangi redundansi, tetapi juga membantu menekan konflik label akibat duplikasi lintas kelas. Ini adalah salah satu langkah cleaning yang paling berdampak langsung terhadap reliabilitas dataset.

In [ ]:
import os
from sklearn.metrics.pairwise import cosine_similarity

print("Memulai proses pembersihan dataset Berdasarkan Skor Similarity Tertinggi")
total_deleted = 0

def hitung_kemiripan_ke_kelas(img_path, nama_kelas):
    """Fungsi untuk menghitung seberapa cocok gambar ini dengan centroid kelasnya saat ini."""
    try:
        idx = valid_paths.index(img_path)
        img_emb = embeddings[idx].reshape(1, -1)

        if nama_kelas in class_centroids:
            centroid = class_centroids[nama_kelas]
            sim = cosine_similarity(img_emb, centroid)[0][0]
            return sim * 100
        else:
            return 0.0
    except ValueError:
        return 0.0

# Looping melalui setiap grup duplikat
for i, group in enumerate(duplicate_groups):
    if len(group) > 1:

        # 1. EVALUASI SEMUA GAMBAR DALAM GRUP TERLEBIH DAHULU
        group_with_scores = []
        for img_path in group:
            kelas_saat_ini = os.path.basename(os.path.dirname(img_path))
            skor_sim = hitung_kemiripan_ke_kelas(img_path, kelas_saat_ini)
            group_with_scores.append({
                'path': img_path,
                'kelas': kelas_saat_ini,
                'skor': skor_sim
            })

        # 2. URUTKAN BERDASARKAN SKOR TERTINGGI (Descending)
        # Gambar yang paling cocok dengan kelasnya akan berada di urutan teratas (index 0)
        group_with_scores.sort(key=lambda x: x['skor'], reverse=True)

        # 3. TETAPKAN PEMENANG DAN YANG HARUS DIHAPUS
        gambar_pemenang = group_with_scores[0]
        gambar_dihapus = group_with_scores[1:]

        print(f"\nGrup {i + 1}:")
        print(f" [+] Mempertahankan : {os.path.basename(gambar_pemenang['path'])} (Kelas: '{gambar_pemenang['kelas']}') (Kemiripan: {gambar_pemenang['skor']:.1f}%)")

        # Eksekusi penghapusan
        for item in gambar_dihapus:
            img_path = item['path']
            try:
                if os.path.exists(img_path):
                    os.remove(img_path)
                    print(f" [-] Dihapus        : {os.path.basename(img_path)} (Kelas: '{item['kelas']}') (Korelasi terhadap kelas: {item['skor']:.1f}%)")
                    total_deleted += 1
                else:
                    print(f" [!] File tidak ditemukan: {os.path.basename(img_path)}")
            except Exception as e:
                print(f" [X] Error menghapus {os.path.basename(img_path)}: {e}")

print(f"\nSelesai! Berhasil menghapus {total_deleted} gambar duplikat.")

## 12) Sinkronisasi Ulang Path dan Embedding Setelah Penghapusan

Setelah file duplikat dihapus dari disk, daftar `valid_paths` dan `embeddings` perlu disinkronkan ulang agar hanya memuat file yang benar-benar masih tersedia. Langkah ini sangat penting karena analisis lanjutan akan menjadi tidak valid bila memori masih merujuk ke file yang sudah tidak ada.

### Ringkasan Sinkronisasi

| Kondisi | Jumlah |
|---|---:|
| Total gambar sebelum cleaning | 1652 |
| Total gambar sesudah cleaning | 1427 |
| Total gambar terhapus | 225 |
| Persentase pengurangan | 13.6% |

Output ini menegaskan bahwa proses cleaning berhasil mengurangi ukuran dataset secara nyata sekaligus menjaga konsistensi antara file di disk dan objek yang dipakai dalam analisis berikutnya.

In [ ]:
print(f"Jumlah data sebelum filter: {len(valid_paths)}")

filtered_paths = []
filtered_embeddings = []

for path, emb in zip(valid_paths, embeddings):
    if os.path.exists(path):
        filtered_paths.append(path)
        filtered_embeddings.append(emb)

valid_paths = filtered_paths
embeddings = np.array(filtered_embeddings)

print(f"Jumlah data setelah filter: {len(valid_paths)}")

## 13) Evaluasi Distribusi Kelas Setelah Cleaning

Setelah deduplikasi selesai, distribusi kelas dihitung ulang untuk melihat dampak cleaning terhadap komposisi dataset. Tahap ini penting karena penghapusan data dapat memengaruhi keseimbangan antar kelas, dan perubahan tersebut perlu dipahami sebelum dataset digunakan untuk training.

### Perbandingan Distribusi Sebelum vs Sesudah Cleaning

| Kelas | Sebelum | Sesudah | Berkurang |
|---|---:|---:|---:|
| realperson | 465 | 390 | 75 |
| fake_unknown | 338 | 308 | 30 |
| fake_mask | 278 | 248 | 30 |
| fake_screen | 229 | 181 | 48 |
| fake_mannequin | 224 | 199 | 25 |
| fake_printed | 118 | 101 | 17 |

### Statistik Sesudah Cleaning

| Metrik | Nilai |
|---|---:|
| Total kelas | 6 |
| Total gambar | 1427 |
| Rata-rata per kelas | 237.83 |
| Standar deviasi | 101.63 |
| Minimum | 101 |
| Maksimum | 390 |

Dibandingkan kondisi awal, standar deviasi turun dari **117,85** menjadi **101,63**, yang berarti distribusi antar kelas menjadi **sedikit lebih rata** setelah pembersihan. Meski demikian, `realperson` tetap menjadi kelas terbesar dan `fake_printed` tetap menjadi kelas terkecil.

📊 **Kesimpulan:** cleaning ini berhasil **mengurangi redundansi dan noise**, tetapi **tidak ditujukan untuk menyeimbangkan kelas secara penuh**. Fokus utamanya tetap pada peningkatan kualitas data.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# 1. Menghitung ulang jumlah gambar per kelas
cleaned_class_counts = {}
for c in train_classes:
    c_dir = os.path.join(train_dir, c)
    # Mengecek jumlah file yang tersisa di dalam folder
    n = len([f for f in os.listdir(c_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp'))])
    cleaned_class_counts[c] = n

# 2. Membuat Pandas Series dan mengurutkannya
cleaned_counts_series = pd.Series(cleaned_class_counts).sort_values(ascending=False)

# 3. Menampilkan ringkasan statistik terbaru
print('\nRingkasan jumlah gambar per kelas (SETELAH DIBERSIHKAN):')
print(cleaned_counts_series.describe())

print('\nJumlah masing-masing kelas:')
print(cleaned_counts_series.head())

# 4. Visualisasi ulang distribusi kelas (Bar Chart)
plt.figure(figsize=(12, 5))
cleaned_counts_series.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Distribusi Jumlah Gambar per Kelas (Train - Setelah Pembersihan Duplikat)')
plt.xlabel('Kelas')
plt.ylabel('Jumlah Gambar')
plt.xticks(rotation=60, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 5. Visualisasi Proporsi Kelas (Pie Chart)
plt.figure(figsize=(10, 8))
# autopct='%1.1f%%' digunakan untuk menampilkan persentase dengan 1 angka di belakang koma
# cmap='Set3' memberikan warna pastel yang berbeda untuk tiap potongan
cleaned_counts_series.plot(kind='pie', autopct='%1.1f%%', startangle=140, cmap='Set3')
plt.title('Proporsi Persentase Gambar per Kelas (Setelah Pembersihan)', pad=20, fontsize=14, fontweight='bold')
plt.ylabel('') # Menghilangkan label sumbu Y agar tampilan lebih bersih
plt.tight_layout()
plt.show()

## Ringkasan Dampak Cleaning

Agar perubahan dataset lebih mudah dibaca, berikut ringkasan dampak cleaning dalam format singkat.

| Aspek | Sebelum | Sesudah | Perubahan |
|---|---:|---:|---:|
| Total gambar train | 1652 | 1427 | -225 |
| Persentase pengurangan | - | - | 13.6% |
| Standar deviasi distribusi kelas | 117.85 | 101.63 | Turun |
| Status duplikasi lintas kelas | Ada | Dikurangi | Membaik |

Secara kuantitatif, proses cleaning berhasil menghapus **225 gambar** dari total **1.652** data train awal. Dampak terpentingnya bukan hanya pengurangan ukuran dataset, tetapi juga perbaikan kualitas sinyal pelatihan karena redundansi dan konflik label menjadi lebih kecil.

## 14) Audit Top-K Anomali per Kelas (Pendekatan Ringan)

Setelah duplikat dibersihkan, notebook melakukan audit lanjutan dengan menampilkan **Top-K gambar yang paling jauh dari centroid kelasnya** untuk setiap kelas. Pendekatan ini praktis karena langsung memfokuskan perhatian pada contoh-contoh yang paling mencurigakan, tanpa harus meninjau seluruh dataset secara manual.

Visualisasi pada bagian ini berguna untuk melihat apakah kandidat anomali benar-benar tampak menyimpang dari pola mayoritas kelasnya. Pada praktiknya, beberapa gambar terlihat sangat berbeda secara gaya visual, bentuk objek, atau konteks tampilannya dibanding anggota kelas lainnya. Temuan seperti ini mengindikasikan bahwa meskipun deduplikasi sudah dilakukan, dataset masih menyimpan **sampel ambigu**, **kelas residu yang heterogen**, atau kemungkinan **label yang kurang konsisten**.

🔍 Tahap ini sangat penting sebagai jembatan antara audit otomatis berbasis embedding dan keputusan manual seperti relabel atau drop selektif.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print("Mengelompokkan data berdasarkan kelas untuk mencari centroid...")
class_data = {c: {'paths': [], 'embeddings': []} for c in train_classes}

for path, emb in zip(valid_paths, embeddings):
    # Mengambil nama kelas dari path file (nama folder parent dari gambar)
    class_name = os.path.basename(os.path.dirname(path))
    if class_name in class_data:
        class_data[class_name]['paths'].append(path)
        class_data[class_name]['embeddings'].append(emb)

anomalies_per_class = {}
top_k_anomalies = 5  # Jumlah maksimal anomali yang ingin dicek per kelas

print("Menghitung jarak anomali untuk setiap kelas...")
for c in train_classes:
    paths = class_data[c]['paths']
    embs = class_data[c]['embeddings']

    # Lewati jika isi kelas terlalu sedikit untuk dicari rata-ratanya
    if len(paths) < 3:
        continue

    embs_array = np.array(embs)

    # 1. Menghitung Centroid (vektor rata-rata kelas)
    centroid = np.mean(embs_array, axis=0).reshape(1, -1)

    # 2. Menghitung Cosine Similarity setiap gambar terhadap Centroid
    similarities = cosine_similarity(embs_array, centroid).flatten()

    # 3. Gabungkan path dan skor, lalu urutkan dari yang PALING KECIL (paling aneh/berbeda)
    path_sim_pairs = list(zip(paths, similarities))
    path_sim_pairs.sort(key=lambda x: x[1]) # Diurutkan secara ascending (dari terendah)

    # 4. Ambil top-K gambar yang paling tidak mirip dengan rata-rata kelasnya
    anomalies_per_class[c] = path_sim_pairs[:top_k_anomalies]

# --- VISUALISASI GAMBAR ANOMALI ---
print("\nMenampilkan gambar yang dicurigai salah kelas...")

for c in train_classes:
    if c not in anomalies_per_class or len(anomalies_per_class[c]) == 0:
        continue

    anomalies = anomalies_per_class[c]
    num_anomalies = len(anomalies)

    fig, axes = plt.subplots(1, num_anomalies, figsize=(4 * num_anomalies, 4))
    fig.suptitle(f"Kandidat Anomali di Kelas: '{c}'", fontsize=14, fontweight='bold')

    if num_anomalies == 1:
        axes = [axes]

    for j, (img_path, sim) in enumerate(anomalies):
        img = mpimg.imread(img_path)
        axes[j].imshow(img)

        # Tampilkan nama file dan skor similarity (semakin rendah skor, semakin aneh)
        title_text = f"{os.path.basename(img_path)}\nSim to Centroid: {sim:.3f}"
        axes[j].set_title(title_text, fontsize=10, color='red' if sim < 0.6 else 'black')
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

## 15) Audit Anomali dengan Threshold Global

Sebagai evaluasi yang lebih menyeluruh, notebook menghitung kembali jumlah anomali pasca-cleaning menggunakan threshold global yang lebih ketat, yaitu **0,5**. Tujuannya adalah mengukur seberapa banyak sampel yang masih berada cukup jauh dari pusat distribusi kelasnya setelah duplikat dihapus.

### Jumlah Anomali per Kelas

| Kelas | Jumlah Anomali |
|---|---:|
| fake_unknown | 223 |
| realperson | 112 |
| fake_mask | 89 |
| fake_mannequin | 80 |
| fake_printed | 24 |
| fake_screen | 14 |

### Ringkasan Audit Global

| Komponen | Nilai |
|---|---:|
| Threshold global | 0.5 |
| Total anomali | 542 |

Distribusi ini sangat informatif. Kelas **`fake_unknown`** menyumbang jumlah anomali terbesar, yang menguatkan dugaan bahwa kelas ini memang berperan sebagai kategori residu dengan keragaman visual tinggi. Sebaliknya, **`fake_screen`** memiliki jumlah anomali paling sedikit, sehingga tampak lebih konsisten setelah proses deduplikasi.

⚠️ **Interpretasi akhir:** cleaning berbasis duplikasi berhasil memperbaiki kualitas dataset secara signifikan, tetapi belum sepenuhnya menyelesaikan persoalan label noise. Oleh karena itu, hasil pada bagian ini menjadi dasar yang kuat untuk tahap lanjutan seperti relabel manual terarah atau penghapusan sampel yang terlalu ambigu.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.metrics.pairwise import cosine_similarity

print("Mengelompokkan data berdasarkan kelas untuk mencari centroid...")
class_data = {c: {'paths': [], 'embeddings': []} for c in train_classes}

for path, emb in zip(valid_paths, embeddings):
    # Mengambil nama kelas dari path file
    class_name = os.path.basename(os.path.dirname(path))
    if class_name in class_data:
        class_data[class_name]['paths'].append(path)
        class_data[class_name]['embeddings'].append(emb)

anomalies_per_class = {}
total_anomalies_detected = 0
anomaly_threshold = 0.5

print(f"Menghitung jarak anomali (Threshold < {anomaly_threshold})...")
for c in train_classes:
    paths = class_data[c]['paths']
    embs = class_data[c]['embeddings']

    if len(paths) < 3:
        continue

    embs_array = np.array(embs)
    centroid = np.mean(embs_array, axis=0).reshape(1, -1)
    similarities = cosine_similarity(embs_array, centroid).flatten()

    # Gabungkan path dan skor, filter HANYA yang di bawah threshold
    path_sim_pairs = list(zip(paths, similarities))
    anomalous_pairs = [pair for pair in path_sim_pairs if pair[1] <= anomaly_threshold]

    # Urutkan dari yang skornya paling rendah (paling parah)
    anomalous_pairs.sort(key=lambda x: x[1])

    if len(anomalous_pairs) > 0:
        anomalies_per_class[c] = anomalous_pairs
        total_anomalies_detected += len(anomalous_pairs)

# --- RINGKASAN HASIL ---
print("\n" + "="*40)
print(f"TOTAL ANOMALI DITEMUKAN: {total_anomalies_detected} gambar")
print("="*40)
for c, anomalies in anomalies_per_class.items():
    print(f" - Kelas '{c}': {len(anomalies)} anomali")

# --- VISUALISASI GAMBAR ANOMALI ---
print("\nMenampilkan gambar yang dicurigai salah kelas...")

for c, anomalies in anomalies_per_class.items():
    num_anomalies = len(anomalies)

    # Mengatur Grid Dinamis (Maksimal 5 gambar per baris)
    cols = 5
    rows = (num_anomalies + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(15, 3.5 * rows))
    fig.suptitle(f"\nAnomali di Kelas: '{c}' ({num_anomalies} gambar)", fontsize=16, fontweight='bold', y=1.02)

    # Meratakan axes agar mudah di-loop, baik 1D maupun 2D
    if type(axes) is np.ndarray:
        axes = axes.flatten()
    else:
        axes = [axes]

    for j in range(len(axes)):
        if j < num_anomalies:
            img_path, sim = anomalies[j]
            img = mpimg.imread(img_path)
            axes[j].imshow(img)

            title_text = f"{os.path.basename(img_path)}\nSim to Centroid: {sim:.3f}"
            axes[j].set_title(title_text, fontsize=10, color='red')

        # Matikan garis tepi sumbu untuk semua grid (termasuk yang kosong)
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

## 16) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Menghubungkan runtime ke Google Drive dilakukan agar dataset hasil cleaning dapat disimpan secara permanen di luar sesi notebook. Langkah ini penting karena proses cleaning yang sudah dilakukan bersifat bernilai tinggi: hasil deduplikasi dan audit tidak ideal bila hilang saat runtime berakhir. Dengan menyimpan hasil kerja ke Drive, dataset yang sudah lebih bersih dapat langsung dipakai pada eksperimen training berikutnya tanpa harus mengulang seluruh proses dari awal.

## 17) Salin Dataset Bersih ke Google Drive

In [ ]:
import shutil

source_dir = train_dir
target_dir = "/content/drive/MyDrive/dataset_relabel"

print("Copy folder ke Google Drive...")

shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)

print("Selesai")

Tahap ini melakukan backup folder train yang telah dibersihkan ke Google Drive agar dapat digunakan kembali pada eksperimen lanjutan. Pada titik ini, dataset sudah melalui beberapa tahapan penting: validasi struktur awal, analisis distribusi kelas, inspeksi visual, ekstraksi embedding, deteksi duplikat, filtering berbasis centroid, sinkronisasi ulang, dan audit anomali pasca-cleaning.

### Status Akhir Dataset

| Komponen | Nilai |
|---|---:|
| Total gambar awal | 1652 |
| Total gambar akhir | 1427 |
| Total duplikat terhapus | 225 |
| Persentase pengurangan | 13.6% |
| Jumlah kelas | 6 |

Secara keseluruhan, notebook ini berhasil menurunkan ukuran dataset train dari **1.652** menjadi **1.427** gambar, yang berarti **225 gambar duplikat** berhasil dihapus. Dampak terpenting dari langkah ini bukan hanya pengurangan jumlah data, tetapi juga peningkatan kualitas sinyal pelatihan karena redundansi dan konflik label lintas kelas menjadi lebih kecil.

## 18) Tahap Selanjutnya

| Prioritas | Tindakan |
|---|---|
| 1 | Relabel manual terarah pada kandidat paling mencurigakan |
| 2 | Drop selektif untuk sampel yang terlalu ambigu |
| 3 | Normalisasi tambahan seperti rotasi atau standarisasi orientasi |
| 4 | Simpan artefak final dataset bersih untuk tahap modeling |

Tujuan akhir dari tahap lanjutan ini adalah memastikan dataset yang dihasilkan benar-benar **minim noise, lebih konsisten antar label, dan lebih siap untuk training model klasifikasi**. Dataset yang telah dibersihkan dan diproses tersebut selanjutnya akan disimpan dalam direktori `dataset_rotated.zip`.